In [2]:
import boto3
import os

# Initialize the Rekognition client
rekognition = boto3.client('rekognition')

# Create a collection
def create_collection(collection_id):
    try:
        response = rekognition.create_collection(CollectionId=collection_id)
        print(f"Collection {collection_id} created. ARN: {response['CollectionArn']}")
    except rekognition.exceptions.ResourceAlreadyExistsException:
        print(f"Collection {collection_id} already exists.")

# Index a face
def index_face(collection_id, image_path, employee_id):
    with open(image_path, 'rb') as image:
        response = rekognition.index_faces(
            CollectionId=collection_id,
            Image={'Bytes': image.read()},
            ExternalImageId=employee_id,
            DetectionAttributes=['ALL']
        )
    print(f"Face indexed for employee {employee_id}")
    return response['FaceRecords'][0]['Face']['FaceId']

# Recognize a face
def recognize_face(collection_id, image_path):
    with open(image_path, 'rb') as image:
        response = rekognition.search_faces_by_image(
            CollectionId=collection_id,
            Image={'Bytes': image.read()},
            MaxFaces=1,
            FaceMatchThreshold=95
        )
    
    if response['FaceMatches']:
        return response['FaceMatches'][0]['Face']['ExternalImageId']
    else:
        return None


In [3]:
import boto3
import os

def create_collection(collection_id):
    try:
        response = rekognition.create_collection(CollectionId=collection_id)
        print(f"Collection {collection_id} created. ARN: {response['CollectionArn']}")
    except rekognition.exceptions.ResourceAlreadyExistsException:
        print(f"Collection {collection_id} already exists.")

# Initialize the Rekognition client
rekognition = boto3.client('rekognition')
collection_id = "StudentFaces"
create_collection(collection_id)

Collection StudentFaces created. ARN: aws:rekognition:ap-south-1:961341519735:collection/StudentFaces


In [ ]:
student_folder = "student_photos"
for filename in os.listdir(student_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        employee_id = os.path.splitext(filename)[0]
        image_path = os.path.join(student_folder, filename)
        index_face(collection_id, image_path, employee_id)

In [ ]:


    # Index faces (add new employees)
    employee_folder = "student_photos"
    for filename in os.listdir(employee_folder):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            employee_id = os.path.splitext(filename)[0]
            image_path = os.path.join(employee_folder, filename)
            index_face(collection_id, image_path, employee_id)

    # Recognize face (mark attendance)
    attendance_photo = "attendance_photo.jpg"
    recognized_employee = recognize_face(collection_id, attendance_photo)
    
    if recognized_employee:
        print(f"Attendance marked for employee: {recognized_employee}")
    else:
        print("No matching face found.")